# 1 · The coffee network (LP)

**Act 1 of the workshop.** A specialty roaster buys green beans from 3 farms, roasts them in 2 roasteries and delivers roasted coffee to cafés in 4 cities. Find the cheapest plan.

| Stage | Places | Rule |
|---|---|---|
| Farms grow green beans | Colombia (CO), Ethiopia (ET), Brazil (BR) | sell at most their harvest |
| Roasteries roast them | Oakland (OAK), Brooklyn (BKN) | roast at most their capacity; roasting keeps 85% of the weight |
| Cafés sell the coffee | Chicago (CHI), Austin (AUS), Denver (DEN), Atlanta (ATL) | each city gets its demand |

Amounts are in tonnes (t). Costs are in **$1,000 per tonne**, so an objective of 1,604 means about $1.6 million.

Run the cells from top to bottom with **Shift + Enter**.

### Set up this notebook

Installs the packages this notebook needs into the running kernel, on Google Colab, Jupyter or VS Code. Already ran `pip install -r requirements.txt`? It just reports they are installed.

In [ ]:
%pip install -q gamspy pandas plotly
# Optional: install your free academic license (otherwise the demo license is used)
# !gamspy install license <your-access-code>

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import gamspy as gp

## Step 1 · The data

The data lives in ordinary pandas DataFrames. Nothing here is GAMSPy yet.

In [ ]:
harvest_df  = pd.DataFrame({"f": ["CO", "ET", "BR"], "t": [150, 120, 200]})
capacity_df = pd.DataFrame({"r": ["OAK", "BKN"], "t": [260, 180]})
demand_df   = pd.DataFrame({"c": ["CHI", "AUS", "DEN", "ATL"], "t": [90, 60, 75, 80]})

# bean price + ocean freight, $1,000 per tonne of green beans
price_df = pd.DataFrame(
    [["CO", "OAK", 4.3], ["CO", "BKN", 4.4],
     ["ET", "OAK", 4.9], ["ET", "BKN", 4.7],
     ["BR", "OAK", 3.6], ["BR", "BKN", 3.4]],
    columns=["f", "r", "cost"])

# delivery to cafés, $1,000 per tonne of roasted coffee
deliver_df = pd.DataFrame(
    [["OAK", "CHI", 1.2], ["OAK", "AUS", 1.0], ["OAK", "DEN", 0.7], ["OAK", "ATL", 1.5],
     ["BKN", "CHI", 0.7], ["BKN", "AUS", 1.2], ["BKN", "DEN", 1.1], ["BKN", "ATL", 0.6]],
    columns=["r", "c", "cost"])
price_df

## Step 2 · Sets and parameters

A `Container` holds everything. **Sets** are the indices from the algebra (f, r, c). **Parameters** are the data, loaded straight from the DataFrames.

> **Gotcha:** GAMSPy names each symbol after its Python variable, and GAMS names ignore case. So a parameter `D` (demand) and one called `d` (delivery) would clash. That's why the costs are called `price` and `deliver` in the code (*p* and *d* in the math).

In [ ]:
m = gp.Container()

f = gp.Set(m, description="farms", records=["CO", "ET", "BR"])
r = gp.Set(m, description="roasteries", records=["OAK", "BKN"])
c = gp.Set(m, description="cafe cities", records=["CHI", "AUS", "DEN", "ATL"])

H = gp.Parameter(m, domain=f, description="harvest (t)", records=harvest_df)
K = gp.Parameter(m, domain=r, description="roasting capacity (t)", records=capacity_df)
D = gp.Parameter(m, domain=c, description="demand (t roasted)", records=demand_df)
price = gp.Parameter(m, domain=[f, r], description="bean + freight cost", records=price_df)
deliver = gp.Parameter(m, domain=[r, c], description="delivery cost", records=deliver_df)
eta = gp.Parameter(m, description="roasting yield", records=0.85)

## Step 3 · The model

Compare each line with the algebra slide:

- harvest: $\sum_r x_{f,r} \le H_f$
- capacity: $\sum_f x_{f,r} \le K_r$
- balance: $\sum_c y_{r,c} \le \eta \sum_f x_{f,r}$
- demand: $\sum_r y_{r,c} \ge D_c$

In [ ]:
x = gp.Variable(m, domain=[f, r], type="positive", description="t of green beans bought")
y = gp.Variable(m, domain=[r, c], type="positive", description="t of roasted coffee shipped")

harvest  = gp.Equation(m, domain=f)
capacity = gp.Equation(m, domain=r)
balance  = gp.Equation(m, domain=r)
demand   = gp.Equation(m, domain=c)

harvest[f]  = gp.Sum(r, x[f, r]) <= H[f]
capacity[r] = gp.Sum(f, x[f, r]) <= K[r]
balance[r]  = gp.Sum(c, y[r, c]) <= eta * gp.Sum(f, x[f, r])
demand[c]   = gp.Sum(r, y[r, c]) >= D[c]

cost = gp.Sum((f, r), price[f, r] * x[f, r]) + gp.Sum((r, c), deliver[r, c] * y[r, c])
coffee = gp.Model(m, equations=m.getEquations(),
                  problem="LP", sense="min", objective=cost)

## Step 4 · Solve and read the plan

In [ ]:
coffee.solve()
print(coffee.status)
print(f"Total cost: {coffee.objective_value:,.1f} ($1,000)")

In [ ]:
print("Bought, x (t green beans)")
display(x.pivot().round(1))
print("Shipped, y (t roasted coffee)")
display(y.pivot().round(1))

**Questions to discuss**

1. Which farm feeds Brooklyn? Why that one?
2. Ethiopia sells only a little. Why?
3. Which roastery is full?

## Step 5 · Marginals are shadow prices

Every constraint comes back with a **marginal**: how much the total cost changes if you loosen that constraint by one unit. A constraint with spare room has marginal 0.

In [ ]:
print("Roasting capacity")
display(capacity.records[["r", "level", "marginal"]])
print("Harvest")
display(harvest.records[["f", "level", "marginal"]])
print("Demand")
display(demand.records[["c", "level", "marginal"]])

### Predict, then run

Write down a guess first: **if Brooklyn could roast 1 more tonne (181 t instead of 180 t), how much would we save?**

Then run the next cell.

In [ ]:
base_cost = coffee.objective_value

K.setRecords([("OAK", 260), ("BKN", 181)])
coffee.solve()
print(f"{base_cost:,.3f}  ->  {coffee.objective_value:,.3f}")
print(f"Saved: {base_cost - coffee.objective_value:.3f} ($1,000)")

K.setRecords(capacity_df)   # put the data back
coffee.solve();

## Step 6 · Models are experiments: a Brazilian drought

Once the model exists, a what-if is a for-loop. Cut Brazil's harvest in 5% steps, re-solve each time, and record the cost and the plan.

In [ ]:
rows = []
for cut in [0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    H.setRecords([("CO", 150), ("ET", 120), ("BR", 200 * (1 - cut))])
    coffee.solve()
    plan = x.pivot().reindex(index=["CO", "ET", "BR"], columns=["OAK", "BKN"]).fillna(0)
    rows.append({"cut %": round(100 * cut), "cost": coffee.objective_value,
                 "ET to BKN (t)": plan.loc["ET", "BKN"]})

H.setRecords(harvest_df)   # put the data back
drought = pd.DataFrame(rows)
drought["extra cost"] = drought["cost"] - drought["cost"].iloc[0]
drought.round(1)

In [ ]:
colors = ["#F4961A" if t > 0 else "#9AA3AE" for t in drought["ET to BKN (t)"]]
fig = go.Figure(go.Bar(x=drought["cut %"], y=drought["extra cost"], width=3.5, marker_color=colors))
fig.update_layout(title="Orange: Brooklyn has to buy Ethiopian beans",
                  xaxis_title="Cut to Brazil's harvest (%)", yaxis_title="Extra cost ($1,000)",
                  template="simple_white", width=800, height=400)
fig.show()

**Why is it a straight line of $13k per step?** Each 5% is 10 t, and the Brazil harvest marginal says each tonne is worth 1.3 ($1,000). The marginal predicted the experiment.

## Step 7 · Look under the hood

The equation listing shows the rows GAMSPy generated from your one line of code. Oakland's balance row is exactly the `-0.85` row of the matrix on the slides.

In [ ]:
coffee.solve(options=gp.Options(equation_listing_limit=10))
print(balance.getEquationListing())

In [ ]:
coffee.toLatex(path="coffee_latex", generate_pdf=False)   # the model as LaTeX math
coffee.toGams(path="coffee_gams")                          # the same model as a GAMS file

## Your turn

**Core**

1. Oakland's roastery breaks down for a day and can only take 150 t. Change `K`, re-solve, and describe what changes in the plan.
2. Chicago's demand doubles to 180 t. Which constraint becomes the bottleneck? (Look at the marginals.)

**Stretch**

3. Add a fourth farm, Vietnam (VN): harvest 100 t, cost 3.9 to OAK and 4.2 to BKN. You only need to change data. Does the model code change at all?
4. A new rule: at least 20% of each roastery's beans must come from Ethiopia (a quality blend). Add this as a new `Equation` and see what it costs.

In [ ]:
# Your work here
